# Storage — Blob, Files, Disks & Tiers

Almost every Azure workload eventually writes to storage. Logs go to a blob. Images land in a CDN-backed container. Backups archive to cold tier. A Spark job reads parquet from Data Lake. A virtual machine boots from a managed disk. Three different services — Blob, Files, and Disks — meet most of those needs, and they all live inside one container: the **storage account**.

The trick to thinking about Azure storage is to separate the three orthogonal choices: the **account** (kind + redundancy + region), the **service** (Blob, File, Queue, Table), and the **tier** (Hot, Cool, Cold, Archive, or the disk performance tiers). Pick badly on any one and you either over-pay or hit a performance cliff.

## The storage account

A storage account is the namespace and security boundary for blobs, files, queues, and tables. The name is globally unique (it backs the URL `https://<name>.blob.core.windows.net`), DNS-safe (3–24 chars, lowercase, numbers).

The **kind** decides what services and SKUs are available:

- **StorageV2 (general purpose v2)** — the default. Supports blob, file, queue, table; all access tiers; all redundancy options. Pick this unless you have a reason not to.
- **Premium block blob** — SSD-backed; sub-10 ms latency for blob workloads with high transaction rates.
- **Premium file shares** — SSD-backed for SMB/NFS file shares that need low latency.
- **Premium page blobs** — used for unmanaged VHDs; almost obsolete since managed disks took over.

An account has its own endpoint per service: `<name>.blob.core.windows.net`, `<name>.file.core.windows.net`, `<name>.queue.core.windows.net`, `<name>.table.core.windows.net`. Each can be locked behind a **private endpoint** so traffic never traverses the public internet.

## Redundancy — where your bytes live

Every storage account picks one **redundancy** SKU at create time. The SKU sets how many copies of your data exist and how widely they are spread.

```
                Copies   Spread                            Survives
LRS             3        one datacenter                    disk / rack failure
ZRS             3        three AZs in one region           whole-DC failure
GRS             6        primary region + paired region    region failure (read-only)
RA-GRS          6        same as GRS, secondary readable   region failure (read)
GZRS            6        ZRS in primary + LRS in paired    DC + region failure
RA-GZRS         6        GZRS with secondary readable      best HA + DR
```

Locally-redundant (LRS) is the cheapest and still survives the most common failure (a disk or rack). Zone-redundant (ZRS) jumps to whole-datacenter tolerance and is the right default for most production data.

The geo variants (GRS/GZRS) add **asynchronous replication** to the paired region — important because the paired region copy lags by seconds to minutes, so a *failover* is not free of data loss. The **RA-** prefix exposes a read-only secondary endpoint that you can hit while the primary is still up, which is useful for read-heavy global apps.

**Customer-managed failover** lets you trigger a primary↔secondary swap yourself rather than waiting for Microsoft. It is a one-way operation that loses the original primary, so test it before you ever need it.

AWS comparison: LRS ≈ S3 in one AZ; ZRS ≈ standard S3; GRS ≈ S3 with cross-region replication; RA-GRS ≈ S3 multi-region access points.

## Blob storage

Blob is Azure's object store and the workhorse of the storage account. Three blob types, chosen at create time:

- **Block blob** — most blobs. Optimised for upload, parallelisable, versioned. Use for documents, images, backups, parquet, logs.
- **Append blob** — write-once-per-block, optimised for log ingestion. Each `Append` is atomic and ordered.
- **Page blob** — random read/write of fixed-size pages. Backs unmanaged VHDs; you rarely create these by hand any more.

Within block blob, each object has an **access tier** that trades latency for storage cost:

| Tier | First-byte latency | Storage cost | Read cost | When |
|------|--------------------|--------------|-----------|------|
| Hot | milliseconds | $$$ | cheap | frequently accessed |
| Cool | milliseconds | $$ | higher | accessed monthly |
| Cold | milliseconds | $ | higher still | accessed quarterly |
| Archive | hours (rehydrate) | $ | expensive | accessed yearly |

Hot, Cool, and Cold are *online* — the first byte returns in milliseconds. **Archive is offline** — you have to issue a `RehydrateBlob` operation that takes hours, during which the blob is unreadable. Treat Archive as tape, not cold storage.

**Lifecycle management** policies move blobs between tiers automatically by age. The classic policy: move to Cool after 30 days, to Cold after 90, to Archive after 180, delete after 7 years. You write the rules in JSON; the storage service evaluates them once per day.

AWS comparison: tiers map directly. Hot ≈ S3 Standard, Cool ≈ Standard-IA, Cold ≈ One Zone-IA / Glacier Instant, Archive ≈ Glacier Flexible / Deep Archive. Lifecycle rules look almost identical.

## Immutability, versioning, soft delete

Three protections for data you cannot afford to lose:

- **Soft delete** keeps deleted blobs and containers around for a configurable retention window (1–365 days). The recovery operation is a single API call. Enable it on every production account.
- **Versioning** keeps every previous version of a blob automatically when you overwrite it. List by version ID; restore by promoting a version back to current. Pairs naturally with soft delete.
- **Immutability policies** make blobs WORM — write-once-read-many. Two modes: **time-based** (locked until a date) and **legal hold** (locked until cleared). Required for many compliance regimes (SEC 17a-4, FINRA). Once a policy is **locked**, even an Owner cannot delete or modify the data.

Combine all three for ransomware resilience: soft delete with a 30-day retention, versioning on every container, and a 1-year time-based immutability policy on backup containers. An attacker with full RBAC still cannot destroy the backups.

## SAS tokens and access control

Three ways to authorise a request to blob storage:

- **Entra ID + RBAC** — the modern default. The caller authenticates with their identity (user, service principal, managed identity) and Azure checks role assignments like `Storage Blob Data Contributor`. No shared secrets to leak.
- **Shared Key** — the account-wide master credential. Two keys per account (for rotation). Anyone with the key has full access to everything. Treat it like a root password and rotate aggressively — better still, disable shared-key access at the account level and force Entra-only.
- **Shared Access Signature (SAS)** — a time-bounded, scope-limited URL token. Three flavours:
  - **User-delegation SAS** — signed by an Entra ID identity (not the shared key). Auditable; revocable by revoking the identity's permissions. The preferred SAS flavour.
  - **Service SAS** — signed by the shared key, scoped to a single service.
  - **Account SAS** — signed by the shared key, scoped to the whole account.

A SAS token in a URL is a bearer credential — anyone who sees it can use it. Always set the shortest practical expiry, restrict by IP if possible, scope to the narrowest container/blob, and prefer user-delegation over service or account SAS.

## Encryption

All Azure Storage is **encrypted at rest by default** with platform-managed AES-256 keys. The choice you make is what *manages* the keys:

- **Microsoft-managed keys (MMK)** — the default. Microsoft generates and rotates the keys; you never see them.
- **Customer-managed keys (CMK)** — keys live in your Key Vault (or Managed HSM). The storage account uses the CMK to wrap an internal data-encryption key. Revoke the CMK and the data is cryptographically inaccessible — a useful compliance lever, and a dangerous foot-gun if you fat-finger a delete.
- **Customer-provided keys (CPK)** — the caller supplies a key per request. Niche; mostly used by storage gateways.

Encryption in transit is HTTPS-only when you flip the **"secure transfer required"** flag at the account level. Always on for production accounts.

## Azure Data Lake Storage Gen2

ADLS Gen2 is *not* a separate service — it is a storage account with the **hierarchical namespace (HNS)** flag flipped on at create. Flipping HNS gives you:

- **Real directories** — rename a directory atomically instead of renaming every blob underneath it. Matters hugely for Spark, Hadoop, and any tool that does `mv /staging/out /final/year=2026/month=06/`.
- **POSIX-style ACLs** on directories and files in addition to RBAC.
- **ABFS driver** support for Hadoop/Spark, which is faster than the legacy `wasbs://` driver.

HNS cannot be turned on or off after the account is created. Decide before you create it. For any analytics workload — Spark, Synapse, Databricks, Fabric — turn HNS on. For pure object-storage workloads (static websites, image hosting), leave it off; the flat namespace is slightly cheaper.

AWS comparison: ADLS Gen2 ≈ S3 with a hierarchical-namespace overlay. There is no direct S3 equivalent — most Spark-on-S3 stacks pay the rename penalty or use Iceberg/Delta to avoid it.

In [ ]:
# Walk a blob workflow end to end.

RG=rg-storage-demo
ST=stfoundations$RANDOM
az group create --name $RG --location eastus

# 1. ZRS storage account with HNS on (so this is ADLS Gen2).
az storage account create \
  --resource-group $RG --name $ST \
  --kind StorageV2 --sku Standard_ZRS \
  --hierarchical-namespace true \
  --allow-blob-public-access false \
  --min-tls-version TLS1_2

# 2. Disable shared-key access — force Entra ID auth.
az storage account update --name $ST --resource-group $RG --allow-shared-key-access false

# 3. Create a container and upload a file using your Entra identity.
az storage container create --name raw --account-name $ST --auth-mode login
az storage blob upload --container raw --file ./data.parquet \
  --name year=2026/month=06/data.parquet --account-name $ST --auth-mode login

# 4. Apply a lifecycle policy: tier to Cool after 30d, Archive after 180d.
az storage account management-policy create --account-name $ST --resource-group $RG \
  --policy '{
    "rules":[{
      "name":"tier-old","enabled":true,
      "type":"Lifecycle",
      "definition":{
        "filters":{"blobTypes":["blockBlob"]},
        "actions":{"baseBlob":{
          "tierToCool":{"daysAfterModificationGreaterThan":30},
          "tierToArchive":{"daysAfterModificationGreaterThan":180}
        }}
      }
    }]
  }'

# 5. Generate a short-lived user-delegation SAS for sharing one blob.
EXPIRY=$(date -u -v+1H +%Y-%m-%dT%H:%MZ)
az storage blob generate-sas \
  --account-name $ST --container-name raw \
  --name year=2026/month=06/data.parquet \
  --permissions r --expiry $EXPIRY \
  --auth-mode login --as-user --https-only

## Azure Files

**Azure Files** is managed shared file storage you mount over SMB or NFS. Two performance tiers:

- **Standard** — backed by HDDs; sized by capacity; pay-as-you-go. For low-throughput general-purpose shares.
- **Premium** — backed by SSDs; provisioned throughput and IOPS scale with provisioned capacity. For high-IO workloads.

Two protocols:

- **SMB 3.x** — Windows and modern Linux/macOS clients. Identity integration via Entra Domain Services or on-prem AD Kerberos. The default.
- **NFS 4.1** — Linux/Unix workloads needing POSIX semantics. Premium-only. No identity integration; authorisation is by network position (NSGs, private endpoints).

**Azure File Sync** is the hybrid feature: install a sync agent on a Windows file server on-prem, point it at an Azure Files share, and the agent **tiers cold files to the cloud** while leaving hot files local. Users see one familiar UNC path; storage costs collapse because rarely-touched files leave the on-prem disk. The pattern is the most common lift-and-shift for departmental file servers.

AWS comparison: SMB shares ≈ FSx for Windows; NFS shares ≈ EFS or FSx for Lustre/ONTAP; File Sync has no direct AWS equivalent.

## Managed disks (from the storage angle)

Notebook 03 covered disks from a VM-sizing perspective. From the storage angle, two operational features matter:

- **Snapshots** are incremental, crash-consistent point-in-time copies stored as standard storage. A snapshot of an unchanged disk is essentially free (only the delta is stored). The right backup primitive for a VM that is too small for full Azure Backup.
- **Shared disks** allow the same managed disk to be attached to multiple VMs at once (with the `sharingOption` set and `maxShares > 1`). Required for SQL Server Failover Cluster Instances, Linux Pacemaker, and other clustered file systems. Available on Premium SSD, Premium SSD v2, and Ultra.

Disk-level encryption inherits the same MMK / CMK choice as the storage account, with the bonus of **encryption-at-host**, which encrypts the host's local cache and the OS disk before it ever hits the storage layer. Turn it on for high-compliance workloads.

## Azure NetApp Files

**Azure NetApp Files (ANF)** is a separate, premium file service for the highest-end workloads: SAP HANA shared storage, large EDA simulations, GPU training data sets, latency-sensitive databases that need NFSv4.1 or SMB at sub-millisecond latency with hundreds of thousands of IOPS per volume.

It is provisioned in **capacity pools** (size + service level: Standard, Premium, Ultra), and you carve **volumes** out of the pool. Snapshots are instantaneous and space-efficient. Cross-region replication is built in.

ANF is meaningfully more expensive than Premium Azure Files and you should only reach for it when the workload genuinely needs it. The trigger is usually a vendor benchmark — SAP, an EDA tool, or an AI training pipeline — that says "NetApp ONTAP at these latencies or we won't certify the deployment."

## Putting it together

A production storage layout, by workload:

- **Application data (parquet, images, backups)** — StorageV2 with HNS on if analytics, ZRS for HA, lifecycle rules tiering by age, soft delete + versioning + immutability for ransomware resilience. Entra ID auth only; shared key disabled.
- **VM disks** — Premium SSD v2 for production OS and data; CMK + encryption-at-host for compliance; snapshots for quick restore.
- **Shared file storage** — Azure Files SMB Premium for Windows estates; NFS Premium for Linux; File Sync for hybrid offices.
- **Highest-performance file** — NetApp Files only when a benchmark forces it.

Get the account, the redundancy, and the tier right, and storage becomes the boring thing it should be — a place to put bytes, not a recurring source of incidents.